# Translated lincRNA ORFs in hepatic cell lines

Automated discovery and prioritization of translated ORFs in de novo annotated lincRNAs of HepG2 / Huh7,
replacing the manual IGV pre-selection used for ZFAS1 and FRG1BP (see [README.md](README.md)).

* **Heavy computations** are done by the Nextflow pipeline [lincorfz](lincorfz/), run as SLURM jobs.
* **This notebook** prepares inputs and parameters, prints the bash/`sbatch` commands, and inspects the results.
  It contains only light-weight code (reading tables, plotting). It can itself be executed as a SLURM job,
  see the last section.
* **All outputs** are written to `<WF_RUNS_DIR>/Liver_lincRNA_ORFs_Claude/`.

Commands printed by this notebook must be executed on a sciCORE **login node** (they only submit SLURM jobs
or install software).

**Conda environment / kernel:**

Use the pipeline's Python environment **`lincorfz_py`** as the Jupyter kernel of `Liver_lincRNA_ORFs.ipynb`
(not the environment of the main repository README). It is built together with the pipeline by
`lincorfz/setup_conda_envs.sh` and lives at
`<WF_RUNS_DIR>Liver_lincRNA_ORFs_Claude/conda_envs/lincorfz_py` (`WF_RUNS_DIR` is defined in the `.env` file).
Register it as a kernel once (e.g. on a login node), then select "lincorfz_py" in Jupyter / VS Code:
```bash
set -a; source ../Liver_projects.scicore.env; set +a     # from Liver_lincRNA_ORFs/
PYTHONNOUSERSITE=1 ${WF_RUNS_DIR}Liver_lincRNA_ORFs_Claude/conda_envs/lincorfz_py/bin/python     -m ipykernel install --user --name lincorfz_py --display-name "lincorfz_py"
```
Alternatively, execute the whole notebook as a SLURM job with `sbatch/execute_notebook.sbatch` (last section of the
notebook), which uses the same environment. The bash/`sbatch` commands printed by the notebook need the
`nextflow` conda environment only for the pipeline driver (activated inside `sbatch/run_lincorfz.sbatch`).

In [ ]:
import os
import re
import json
import glob
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

sns.set_theme(style="white", font_scale=1.0)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

In [ ]:
# 1. environment variables (shared Liver_projects configuration, symlinked into the repository root)
load_dotenv("../Liver_projects.scicore.env")

subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "ext_data_dir": os.getenv("EXT_DATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
}
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
}

# 2. project-specific locations
PROJECT_DIR = Path.cwd()                      # .../CRISPR_projects/Liver_lincRNA_ORFs
LINCORFZ_DIR = PROJECT_DIR / "lincorfz"       # Nextflow pipeline
SBATCH_DIR = PROJECT_DIR / "sbatch"

OUT_BASE = Path(subdirs["wf_runs_dir"]) / "Liver_lincRNA_ORFs_Claude"   # ALL outputs go here
INPUTS_DIR = OUT_BASE / "inputs"
PARAMS_DIR = OUT_BASE / "params"
RUNS_DIR = OUT_BASE / "runs"
CONDA_ENVS_DIR = OUT_BASE / "conda_envs"
FIG_DIR = OUT_BASE / "figures"
for d in [INPUTS_DIR, PARAMS_DIR, RUNS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# data sources
RNASEQ_RUN_DIR = Path(subdirs["wf_runs_dir"]) / "Asiers_HepG2_Huh7_RNAseq_v1" / "output"
RIBOSEQ_BAM_DIR = Path(subdirs["lab_group_dir"]) / ("RNA_oxidation/wf_runs/Juan_pre_rRNA/results/"
                   "splicing_adapter_polya4_local_sjdb_annot3_novel8_NM1_all_replicates_v1/resolution")
POLYSOME_DIR = Path(subdirs["wf_runs_dir"]) / "polysome_seq_hepg2_inayat_v1" / "polysome_normalised_biwig_data"
CONSERVATION_DIR = Path(subdirs["ext_data_dir"]) / "conservation_and_mutations"
HCC_ANNOT_DIR = Path(subdirs["tables_dir"]) / "HCC_ORFs" / "annotations"
print("outputs:", OUT_BASE)

def write_if_changed(text, path):
    # Write `text` to `path` only if the content differs: unchanged files keep their modification
    # time, so that the -resume cache of Nextflow is not invalidated by re-running this notebook.
    path = Path(path)
    if path.exists() and path.read_text() == text:
        return False
    path.write_text(text)
    return True

## 1. Conda environments

sciCORE compute nodes have no internet access, therefore the two environments of the pipeline
(`lincorfz/envs/*.yml`) are built **once, on a login node**; Nextflow then only activates them inside the
SLURM jobs. The `nextflow` conda environment (Nextflow >= 24) is used to run the pipeline driver.

In [ ]:
print(f"bash {LINCORFZ_DIR}/setup_conda_envs.sh {CONDA_ENVS_DIR}")
print("# if not available yet:  conda create -n nextflow -c conda-forge -c bioconda nextflow")

## 2. Prepare inputs

Sample sheets read by the pipeline:

* `rnaseq.tsv`: HepG2 / Huh7 bulk RNA-seq (PMID 40885748): STAR genome alignments and `SJ.out.tab`
* `riboseq.tsv`: HepG2 CHX-only Ribo-seq, genomic alignments with resolved multimappers
* `polysome.tsv`: HepG2 polysome fractions 80S (k = 1 ribosome) ... decasome (k = 10); the free, 40S, 60S and
  heavysome fractions are not used for the mean ribosome load
* `rnaseq_tracks.tsv`: library-size normalized RNA-seq bigwigs (per cell line and strand), only for the IGV session

In [ ]:
# RNA-seq: sample ids and cell lines from the SRA run table
sra = pd.read_csv(Path(subdirs["metadata_dir"]) / "SraRunTable.PMID40885748.csv")
rnaseq = pd.read_csv(Path(subdirs["metadata_dir"]) / "start_samples.rnaseq.pe.tsv", sep="\t")[["sample"]]
rnaseq = rnaseq.merge(sra[["Run", "cell_line"]].rename(columns={"Run": "sample", "cell_line": "condition"}), on="sample")
rnaseq["bam"] = [str(RNASEQ_RUN_DIR / "samples" / s / "map_genome" / f"{s}.Aligned.sortedByCoord.out.bam") for s in rnaseq["sample"]]
rnaseq["sj_tab"] = [str(RNASEQ_RUN_DIR / "samples" / s / "map_genome" / f"{s}.SJ.out.tab") for s in rnaseq["sample"]]
assert all(os.path.exists(p) for p in list(rnaseq["bam"]) + list(rnaseq["sj_tab"]) + [b + ".bai" for b in rnaseq["bam"]])
rnaseq = rnaseq.sort_values(["condition", "sample"])
write_if_changed(rnaseq.to_csv(sep="\t", index=False), INPUTS_DIR / "rnaseq.tsv")
display(rnaseq[["sample", "condition"]])

In [ ]:
# Ribo-seq: HepG2, cycloheximide only, 3 replicates
riboseq = pd.DataFrame({"sample": [f"trplct_{i}_HepG2_CHXonly" for i in (7, 8, 9)]})
riboseq["bam"] = [str(RIBOSEQ_BAM_DIR / f"{s}.resolved.bam") for s in riboseq["sample"]]
assert all(os.path.exists(p) and os.path.exists(p + ".bai") for p in riboseq["bam"])
write_if_changed(riboseq.to_csv(sep="\t", index=False), INPUTS_DIR / "riboseq.tsv")
display(riboseq)

In [ ]:
# polysome fractions -> number of ribosomes; strands: forward = plus, reverse = minus
n_ribosomes = {"80s": 1, "disome": 2, "trisome": 3, "tetrasome": 4, "pentasome": 5,
               "hexasome": 6, "heptasome": 7, "octasome": 8, "nonasome": 9, "decasome": 10}
rows = []
for p in sorted(POLYSOME_DIR.glob("*.bw")):
    m = re.match(r"HepG2polysome(.+)_rep(\d+)\.spikeinNorm\.(forward|reverse)\.bw", p.name)
    fraction, rep, s = m.groups()
    if fraction in n_ribosomes:
        rows.append(dict(path=str(p), fraction=fraction, n_ribosomes=n_ribosomes[fraction], replicate=int(rep),
                         strand="plus" if s == "forward" else "minus"))
polysome = pd.DataFrame(rows).sort_values(["n_ribosomes", "replicate", "strand"])
write_if_changed(polysome.to_csv(sep="\t", index=False), INPUTS_DIR / "polysome.tsv")
display(polysome.groupby(["n_ribosomes", "fraction"])["replicate"].apply(lambda x: sorted(set(x))).to_frame("replicates"))

In [ ]:
# RNA-seq coverage (library-size normalized, per cell line and strand) for the IGV session
ls_dir = RNASEQ_RUN_DIR / "LS_normalized_genomic_coverage" / "per_etiology_group"
rnaseq_tracks = pd.DataFrame([dict(label=c, strand="+" if s == "plus" else "-", path=str(ls_dir / f"{s}.{c}.bw"))
                              for c in ["HepG2", "Huh7"] for s in ["plus", "minus"]])
assert all(os.path.exists(p) for p in rnaseq_tracks["path"])
write_if_changed(rnaseq_tracks.to_csv(sep="\t", index=False), INPUTS_DIR / "rnaseq_tracks.tsv")

## 3. Parameter files

Only inputs and run locations are set here; all analysis parameters keep the defaults documented in
`lincorfz/nextflow.config` unless overridden below. `test_chr20` restricts the analysis to chromosome 20,
which carries both previously selected targets (ZFAS1, FRG1BP).

In [ ]:
common_params = {
    "rnaseq_tsv": str(INPUTS_DIR / "rnaseq.tsv"),
    "riboseq_tsv": str(INPUTS_DIR / "riboseq.tsv"),
    "polysome_tsv": str(INPUTS_DIR / "polysome.tsv"),
    "genome": file_paths["human_genome_file"],
    "chrom_sizes": file_paths["human_chrom_sizes_file"],
    "reference_gtf": file_paths["human_annotation_file"],          # GENCODE v42 comprehensive
    "conservation_dir": str(CONSERVATION_DIR),
    "noncode_gtf": str(HCC_ANNOT_DIR / "NONCODEv6_human_hg38_lncRNA.gtf"),
    "hcc_orfs_gtf": str(HCC_ANNOT_DIR / "nORFs_CDSs.gtf"),          # nORFs of the HCC Ribo-seq paper
    "rnaseq_tracks_tsv": str(INPUTS_DIR / "rnaseq_tracks.tsv"),
    "igv_reference_annotation": str(Path(subdirs["human_annotation_dir"]) / "hg38_v42" / "gencode.v42.annotation.sorted.gtf"),
    "conda_envs_dir": str(CONDA_ENVS_DIR),
}
runs = {
    "test_chr20": {"chromosomes": "chr20"},
    "v1": {"chromosomes": "all"},   # first genome-wide run (before per-read-length tracks and Bayesian test)
    "v2": {"chromosomes": "all"},   # current version
}
for run_name, extra in runs.items():
    p = {**common_params, **extra, "rundir": str(RUNS_DIR / run_name)}
    write_if_changed(json.dumps(p, indent=4), PARAMS_DIR / f"{run_name}.json")
    print(PARAMS_DIR / f"{run_name}.json")

## 4. Run the pipeline

The Nextflow driver runs inside a SLURM job (`sbatch/run_lincorfz.sbatch`, QoS `1day`) and submits every
pipeline task as a separate SLURM job. Re-submitting the same command resumes an interrupted run.

In [ ]:
def launch_command(run_name):
    rundir = RUNS_DIR / run_name
    logs = rundir / "slurm_logs"
    logs.mkdir(parents=True, exist_ok=True)   # SLURM silently fails if the log directory does not exist
    return (f"sbatch --output={logs}/driver_%j.out --error={logs}/driver_%j.err "
            f"{SBATCH_DIR}/run_lincorfz.sbatch {PARAMS_DIR / (run_name + '.json')} {rundir} {LINCORFZ_DIR}")

for run_name in runs:
    print(f"# {run_name}\n{launch_command(run_name)}\n")

In [ ]:
# monitoring (login node)
for run_name in runs:
    rundir = RUNS_DIR / run_name
    print(f"# {run_name}")
    print(f"squeue -u $USER")
    print(f"tail -n 30 {rundir}/.nextflow.log")
    print(f"ls -t {rundir}/slurm_logs | head -2\n")

## 5. Results

Set `RUN_NAME` to the run to inspect. Figures are saved to `<OUT_BASE>/figures/<RUN_NAME>/`.

In [ ]:
RUN_NAME = "v2"
RES = RUNS_DIR / RUN_NAME / "results"
FIG = FIG_DIR / RUN_NAME
FIG.mkdir(parents=True, exist_ok=True)
print(RES, RES.exists())

### 5.1 P-site offsets of genomic Ribo-seq reads

Offsets are calibrated per read class (aligned length x 5' soft clip). The metagene shows P-sites of all
calibrated reads around annotated start and stop codons of canonical CDSs (red: frame 0).

In [ ]:
offsets = pd.concat([pd.read_csv(f, sep="\t").assign(sample=Path(f).name.split(".")[0])
                     for f in sorted(glob.glob(str(RES / "riboseq/calibration/*.offsets.tsv")))])
ok = offsets.loc[offsets["pass"]]
summary = ok.groupby("sample").agg(classes=("offset", "size"), cds_reads=("n_cds", "sum"),
                                   frame0=("frame0", "sum"))
summary["frac_frame0"] = summary["frame0"] / summary["cds_reads"]
display(summary)
display(ok.pivot_table(index=["aligned_len"], columns=["sample", "clip5"], values="offset"))

meta = pd.concat([pd.read_csv(f, sep="\t").assign(sample=Path(f).name.split(".")[0])
                  for f in sorted(glob.glob(str(RES / "riboseq/calibration/*.metagene.tsv")))])
samples = sorted(meta["sample"].unique())
fig, axes = plt.subplots(len(samples), 2, figsize=(13, 2.6 * len(samples)), squeeze=False)
for i, s in enumerate(samples):
    for j, anchor in enumerate(["start", "stop"]):
        m = meta.loc[(meta["sample"] == s) & (meta["anchor"] == anchor) & meta["position"].between(-30, 45)]
        colors = ["tab:red" if p % 3 == 0 else ("tab:blue" if p % 3 == 1 else "tab:green") for p in m["position"]]
        axes[i, j].bar(m["position"], m["psites"], color=colors, width=0.9)
        axes[i, j].set(title=f"{s}: P-sites around {anchor} codon", xlabel="position (nt)", ylabel="P-sites")
fig.tight_layout()
fig.savefig(FIG / "psite_metagene.pdf")

stats = pd.concat([pd.read_csv(f, sep="\t").set_index("metric")["value"].rename(Path(f).name.split(".")[0])
                   for f in sorted(glob.glob(str(RES / "riboseq/psite_stats/*.psite_stats.tsv")))], axis=1)
display(stats.loc[[i for i in stats.index if not i.startswith("psites_chr")]])

### 5.1b Frame preservation per read length and frame priors

P-sites are tracked per aligned read length (25-36 nt). For each length, the frame distribution of
P-sites in canonical CDS bodies (translated) and 3'UTRs (untranslated background) is quantified, and
Dirichlet-multinomial priors are fitted on ORF-sized windows (20 codons) of CDS bodies and 3'UTRs.
These priors weight each read length by its frame preservation in the Bayesian in-frame test (5.5b).
The per-length P-site tracks (CPM per sample and pooled) are in `riboseq/psite_bigwigs/by_read_length/`.

In [ ]:
pri = pd.read_csv(RES / "riboseq/frame_priors/frame_priors.tsv", sep="\t")
display(pri.loc[pri["dataset"] == "pooled", ["length", "cds_counts", "cds_frac_f0", "cds_frac_f1", "cds_frac_f2",
                                              "utr_frac_f0", "prior_inframe_mean", "prior_concentration",
                                              "null_concentration", "n_windows_cds", "n_windows_utr"]].round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for d, g in pri.groupby("dataset"):
    ls = "-" if d == "pooled" else ":"
    axes[0].plot(g["length"], g["cds_counts"] / 1e6, ls=ls, marker="o", label=d)
    axes[1].plot(g["length"], g["cds_frac_f0"], ls=ls, marker="o", label=d)
    axes[2].plot(g["length"], g["prior_concentration"], ls=ls, marker="o", label=d)
axes[1].axhline(1 / 3, color="grey", lw=0.5)
axes[0].set(xlabel="aligned read length (nt)", ylabel="CDS P-sites (millions)")
axes[1].set(xlabel="aligned read length (nt)", ylabel="CDS P-sites in frame 0", ylim=(0.2, 0.8))
axes[2].set(xlabel="aligned read length (nt)", ylabel="prior concentration (20-codon windows)")
axes[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "frame_by_read_length.pdf")

### 5.2 Mean ribosome load

Isoform-level MRL (coverage-weighted over all exons) of the de novo lincRNAs compared with
Ensembl-canonical protein-coding mRNAs.

In [ ]:
ref_mrl = pd.read_csv(RES / "polysome_scores/polysome.reference_mRNAs.tsv", sep="\t")
tx_mrl = pd.read_csv(RES / "polysome_scores/polysome.transcripts.tsv", sep="\t")
print(f"mRNAs with MRL: {ref_mrl['MRL'].notna().sum()} / {len(ref_mrl)}, median {ref_mrl['MRL'].median():.2f}")
print(f"lincRNA isoforms with MRL: {tx_mrl['MRL'].notna().sum()} / {len(tx_mrl)}, median {tx_mrl['MRL'].median():.2f}")
display(ref_mrl.loc[ref_mrl["gene_name"].isin(["GAPDH", "ACTB", "EEF1A1", "ALB", "APOA1", "TPT1", "RPL3", "ATF4"]),
                    ["gene_name", "MRL", "polysome_mean_coverage", "frac_polysomal_ge2"]])

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
bins = np.linspace(1, 10, 37)
ax.hist(ref_mrl["MRL"].dropna(), bins=bins, density=True, histtype="step", lw=2, label="canonical mRNAs")
ax.hist(tx_mrl["MRL"].dropna(), bins=bins, density=True, histtype="step", lw=2, label="de novo lincRNA isoforms")
ax.set(xlabel="mean ribosome load (80S..decasome)", ylabel="density")
ax.legend()
fig.savefig(FIG / "mrl_distributions.pdf", bbox_inches="tight")

### 5.3 De novo lincRNA isoforms

Reasons for (non-)selection of the assembled transcripts that do not encode annotated proteins.

In [ ]:
cand = pd.read_csv(RES / "lincRNAs/lincRNA_candidates.all.tsv", sep="\t", low_memory=False)
sel = pd.read_csv(RES / "lincRNAs/lincRNA_transcripts.tsv", sep="\t", low_memory=False)
display(cand["selection"].value_counts().to_frame("transcripts"))
sel["category"] = np.select([sel["lncRNA_gene_names"].notna(), sel["pseudogene_names"].notna()],
                            ["GENCODE lncRNA", "pseudogene"], "novel")
print(f"{len(sel)} isoforms in {sel['gene_id'].nunique()} loci")
display(pd.crosstab(sel["category"], sel["n_exons"].clip(upper=5).astype(str).replace("5", ">=5")))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for c in ["GENCODE lncRNA", "pseudogene", "novel"]:
    v = np.log10(sel.loc[sel["category"] == c, "TPM_max"])
    axes[0].hist(v, bins=40, histtype="step", lw=2, label=f"{c} (n={len(v)})")
axes[0].set(xlabel="log10 TPM (best cell line)", ylabel="isoforms")
axes[0].legend()
axes[1].hist(sel["tx_length"].clip(upper=10000), bins=50, color="grey")
axes[1].set(xlabel="isoform length (nt, clipped at 10 kb)", ylabel="isoforms")
fig.tight_layout()
fig.savefig(FIG / "lincRNA_isoforms.pdf")

### 5.4 Expression of lincRNA loci (pySanity)

In [ ]:
expr = pd.read_csv(RES / "expression/lincRNA_loci.expression.tsv", sep="\t")
fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
ax.scatter(expr["log2CPM_HepG2"], expr["log2CPM_Huh7"], s=4, alpha=0.5, color="k")
lim = [np.nanmin(expr[["log2CPM_HepG2", "log2CPM_Huh7"]].values), np.nanmax(expr[["log2CPM_HepG2", "log2CPM_Huh7"]].values)]
ax.plot(lim, lim, ls="--", color="grey")
ax.set(xlabel="HepG2 log2 CPM (pySanity)", ylabel="Huh7 log2 CPM (pySanity)", title=f"{len(expr)} lincRNA loci")
fig.savefig(FIG / "lincRNA_expression_HepG2_vs_Huh7.pdf", bbox_inches="tight")

### 5.5 ORFs: evidence classes and ranking

`ribo_evidence`: strong > moderate > weak > none (see README). Multimapping reads were resolved upstream (one
alignment per read), so they count towards the class; `resolved_multimapper_dominated` flags ORFs whose P-sites
come mostly (> 90%) from reads with MAPQ < 255, i.e. reads that originally had several loci (paralogs, parent
genes of processed pseudogenes). The relationship between read uniqueness and pseudogene annotation is shown below.

In [ ]:
orfs = pd.read_csv(RES / "final/lincORFs.ranked.tsv", sep="\t", low_memory=False)
display(orfs["ribo_evidence"].value_counts().to_frame("ORFs"))
display(orfs.groupby("ribo_evidence")["locus_id"].nunique().to_frame("loci"))
cols = ["rank", "orf_label", "ribo_evidence", "priority_score", "chrom", "orf_start0", "orf_end1", "strand", "n_aa",
        "start_codon", "kozak_context", "MRL", "MRL_pct_vs_mRNA", "log2CPM_HepG2", "log2CPM_Huh7", "pooled_f0",
        "pooled_frac_inframe", "bayes_log10BF_pooled", "n_reps_bayes_support", "support_read_length_range",
        "pooled_p_wilcoxon", "n_reps_p05", "island_log2ratio", "frac_psites_unique", "gene_name_source",
        "resolved_multimapper_dominated", "ribo_evidence_unique_reads", "PhyloCSF_Median", "lncRNA_gene_names",
        "pseudogene_names", "pseudogene_types", "orf_overlaps_small_RNA", "HCC_paper_same_stop"]
cols = [c for c in cols if c in orfs.columns]
display(orfs[cols].head(40))

In [ ]:
orfs["annotation"] = np.select([orfs["pseudogene_types"].astype(str).str.contains("processed_pseudogene")
                                & ~orfs["pseudogene_types"].astype(str).str.contains("unprocessed"),
                                orfs["pseudogene_names"].notna(), orfs["lncRNA_gene_names"].notna()],
                               ["processed pseudogene", "other pseudogene", "GENCODE lncRNA"], "novel")
ev_all = orfs.loc[orfs["ribo_evidence"] != "none"]
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
sns.histplot(data=ev_all, x="frac_psites_unique", hue="annotation", bins=20, multiple="stack", ax=ax)
ax.set(xlabel="fraction of ORF P-sites from MAPQ 255 reads", ylabel="ORFs with Ribo-seq evidence")
display(pd.crosstab(orfs["ribo_evidence"], orfs["resolved_multimapper_dominated"]))
fig.savefig(FIG / "psite_uniqueness_by_annotation.pdf", bbox_inches="tight")

comp = [c for c in ["c_mrl", "c_ribo", "c_island", "c_expr", "c_cons", "c_unique"] if ev_all[c].nunique() > 1]
ev = orfs.loc[orfs["ribo_evidence"].isin(["strong", "moderate", "weak"])]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(ev[comp].corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[0])
axes[0].set(title="score components (Spearman, ORFs with Ribo-seq evidence)")
for e in ["strong", "moderate", "weak"]:
    axes[1].hist(orfs.loc[orfs["ribo_evidence"] == e, "priority_score"], bins=30, histtype="step", lw=2, label=e)
axes[1].set(xlabel="priority score", ylabel="ORFs")
axes[1].legend(title="Ribo-seq evidence")
fig.tight_layout()
fig.savefig(FIG / "score_components.pdf")

### 5.5b Bayesian in-frame test

Per ORF, P-site counts per frame and read length are compared between translation in the ORF frame
(prior from CDS windows) and the mixture of untranslated background (3'UTR windows) and translation in
another frame. Below: agreement with the per-codon Wilcoxon test of the HCC paper, the Bayes factor
distribution, and the per-read-length support of selected ORFs.

In [ ]:
display(pd.crosstab(orfs["ribo_evidence_periodicity_test"], orfs["ribo_evidence_bayes"], margins=True))
fig, ax = plt.subplots(1, 1, figsize=(6, 3.8))
for e in ["strong", "moderate", "weak", "none"]:
    v = orfs.loc[orfs["ribo_evidence"] == e, "bayes_log10BF_pooled"].dropna().clip(-20, 40)
    if len(v):
        ax.hist(v, bins=60, histtype="step", lw=2, label=f"{e} (n={len(v)})")
ax.set(xlabel="pooled log10 Bayes factor (in frame vs. untranslated / other frame)", ylabel="ORFs", yscale="log")
ax.legend(fontsize=8)
fig.savefig(FIG / "bayes_factor_distribution.pdf", bbox_inches="tight")

bayes = pd.read_csv(RES / "orfs/orf_frame_bayes.tsv", sep="\t")
show = orfs.loc[orfs["locus_name"].astype(str).str.contains("ZFAS1|LINC00261"), "orf_id"].head(4)
display(bayes.loc[bayes["orf_id"].isin(show), ["orf_id", "bayes_log10BF_pooled", "bayes_best_model_pooled",
                                                "modal_read_length", "support_read_length_range",
                                                "read_lengths_with_inframe_evidence", "pooled_psites_by_length",
                                                "pooled_frac_inframe_by_length", "pooled_log10BF_by_length"]].T)

### 5.6 Positive controls: ZFAS1 and FRG1BP

Both were selected manually for knock-down / CRISPR excision experiments. ZFAS1 ORF3 (13 codons incl. stop)
has peptide support in public HCC cell line proteomics.

In [ ]:
for name in ["ZFAS1", "FRG1BP"]:
    hit = orfs.loc[orfs["locus_name"].astype(str).str.contains(name)
                   | orfs["lncRNA_gene_names"].astype(str).str.contains(name)
                   | orfs["pseudogene_names"].astype(str).str.contains(name)]
    print(f"{name}: {len(hit)} ORFs, best rank {hit['rank'].min() if len(hit) else 'NA'} of {len(orfs)}")
    display(hit[cols].head(10))

### 5.7 Comparison with the manually selected regions

Regions of interest stored in the IGV session used for the manual screen
(`IGV/Asiers_PolySomeSeq.Riboseq.xml`) are compared with the ORFs called here.

In [ ]:
import xml.etree.ElementTree as ET
session = Path(subdirs["main_project_dir"]) / "IGV" / "Asiers_PolySomeSeq.Riboseq.xml"
roi = pd.DataFrame([dict(chrom=r.get("chromosome"), start0=int(r.get("start")), end1=int(r.get("end")))
                    for r in ET.parse(session).getroot().iter("Region")])
def best_overlap(r):
    o = orfs.loc[(orfs["chrom"] == r["chrom"]) & (orfs["orf_start0"] < r["end1"]) & (orfs["orf_end1"] > r["start0"])]
    if o.empty:
        return pd.Series({"n_orfs": 0, "best_rank": np.nan, "best_orf": "", "evidence": "", "locus": ""})
    b = o.sort_values("rank").iloc[0]
    return pd.Series({"n_orfs": len(o), "best_rank": b["rank"], "best_orf": b["orf_label"],
                      "evidence": b["ribo_evidence"], "locus": b["locus_name"]})
roi = pd.concat([roi, roi.apply(best_overlap, axis=1)], axis=1)
if RUN_NAME.startswith("test_"):
    roi = roi.loc[roi["chrom"].isin(set(orfs["chrom"]))]
display(roi)

### 5.8 Overlap with lincRNA ORFs from HCC tissue Ribo-seq (paper)

In [ ]:
if "HCC_paper_lincRNA_ORFs" in orfs.columns:
    orfs["in_HCC_paper"] = orfs["HCC_paper_lincRNA_ORFs"].notna()
    display(pd.crosstab(orfs["ribo_evidence"], [orfs["in_HCC_paper"], orfs["HCC_paper_same_stop"]],
                        margins=True))

## 6. Browsing the candidates in IGV

Open the session in IGV desktop (File > Open Session). It loads:
`lincORFs.igv.gtf` (de novo lincRNA isoforms + ORFs, `transcript_id` = `ORF<rank>_<locus>_<length>aa`),
the ORF BED (colour = evidence class), GENCODE v42, the MRL (fixed 1-10 scale) and polysomal coverage tracks,
pooled Ribo-seq P-sites (CPM), P-sites per read length (pooled CPM; one overlay per strand, one colour per
length), HepG2/Huh7 RNA-seq coverage and the PhyloCSF tracks; + strand in blue, - strand in red.
Per-sample per-length tracks are in `riboseq/psite_bigwigs/by_read_length/<sample>.len<L>.<strand>.cpm.bw`.
Clicking on an ORF shows all its metrics, including the amino-acid sequence.
To step through the candidates in rank order, load `lincORFs.loci.ranked.tsv` coordinates or use
`lincORFs.orfs.bed` as Region Navigator input (Regions > Import Regions).

In [ ]:
print(RES / "final" / "lincorfz.igv_session.xml")
print(RES / "final" / "lincORFs.igv.gtf")

## 7. Optional: peptide evidence from small-peptidome data

Non-tryptic (e.g. peptidome or HLA-ligandome) mass-spectrometry data can be searched against
`results/final/lincORFs.faa` together with the canonical human proteome. The resulting peptide tables
(any TSV/CSV with a `Peptide`/`Sequence` column, e.g. FragPipe `peptide.tsv`) are passed to the pipeline as
`"peptide_tables": "/path/a.tsv,/path/b.tsv"`; peptides absent from the GENCODE proteome are matched to
the ORFs and add the peptide component to the priority score. Re-submitting the run with `-resume`
recomputes only the affected steps.

Candidate public non-tryptic datasets (to be checked for suitability before use):
* [PXD023143](https://proteomecentral.proteomexchange.org/dataset/PXD023143): HLA immunopeptidome of isolated
  patient-derived hepatocytes (HBV infection, HCC); already part of the PRIDE re-analysis of the HCC paper,
  but there searched with tryptic settings.
* HCC immunopeptidomics used by [Camarena et al., Sci Adv 2024](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11235171/)
  (ref. 31 of the HCC paper), which reported lncRNA-derived HLA ligands in HCC; see their data availability section.

Such data must be searched with unspecific cleavage (e.g. FragPipe "Nonspecific-HLA" workflow) against
`lincORFs.faa` + the human proteome.

In [ ]:
# example: add peptide tables to the v1 parameters and resume
# p = json.load(open(PARAMS_DIR / "v1.json"))
# p["peptide_tables"] = ",".join(["/path/to/peptidome1/peptide.tsv", "/path/to/peptidome2/peptide.tsv"])
# json.dump(p, open(PARAMS_DIR / "v1.json", "w"), indent=4)
# print(launch_command("v1"))

## 7.1 Peptide evidence from HepG2 total proteomics (MassIVE MSV000096514)

Total proteomics of the same HepG2 cultures as Ribo-seq and polysome profiling (DMEM, time 0; Scientific Data
2025, 12:1520, Fig. 1a): runs `001/002/003_HP-0057_D23` (Supplementary Table S1, condition `HepG2_DMEM`,
replicates 1-3). DIA on an Orbitrap Fusion Lumos, tryptic digest.

The raw files were searched with the lab's FragPipe Snakemake workflow (`zavolanlab/fragpipe-snakemake`,
FragPipe 24.0, template `DIA_SpecLib_Quant`: MSFragger-DIA, spectral library, DIA-NN quantification) against
the UniProt human proteome plus the protein sequences of **all** ORFs called by lincorfz (identical sequences
merged into one entry; decoys and contaminants are added by the workflow). Default FDR filters (1%).

A peptide counts as evidence for a lincRNA ORF only if it does not occur in any canonical human or contaminant
protein, nor in the translation of any GENCODE v42 CDS (all isoforms; I and L treated as identical). Many lincorfz ORFs lie in pseudogenes whose sequences are close to
the parent protein; peptides shared with the parent are reported but are not evidence.

This section is an annotation only: it writes new tables under `proteomics/analysis/` and does **not** change
the ranking or the priority score of the pipeline run.

In [ ]:
PROT_DIR = OUT_BASE / "proteomics"
FP_DIR = PROT_DIR / "fragpipe-snakemake"          # working copy of zavolanlab/fragpipe-snakemake (config.yaml)
FP_OUT = PROT_DIR / "fragpipe_out"
PROT_ANA = PROT_DIR / "analysis"
PROT_ANA.mkdir(parents=True, exist_ok=True)
RAW_RUNS = ["001_HP-0057_D23", "002_HP-0057_D23", "003_HP-0057_D23"]   # HepG2 DMEM time 0, Table S1

# bash (VS Code/login node, needs internet): download raw files over HTTPS (the FTPS data channel of
# massive-ftp.ucsd.edu is reset from the cluster), build the search FASTA, pre-build the converter conda env
url = "https://massive.ucsd.edu/ProteoSAFe/DownloadResultFile?file=f.MSV000096514/raw/raw/{f}&forceDownload=true"
print(f"mkdir -p {PROT_DIR}/raw && cd {PROT_DIR}/raw")
for r in RAW_RUNS:
    print(f"curl -s --fail -o {r}.raw '{url.format(f=r + '.raw')}'")
print(f"cd {FP_DIR} && snakemake --use-conda --conda-create-envs-only --cores 1 "
      f"{FP_OUT}/dia-quant-output/report.pg_matrix.tsv")
# bash (submits the Snakemake driver; every rule runs as its own SLURM job)
print(f"mkdir -p {PROT_DIR}/slurm_logs && sbatch --output={PROT_DIR}/slurm_logs/driver_%j.out "
      f"--error={PROT_DIR}/slurm_logs/driver_%j.err {SBATCH_DIR}/run_fragpipe_snakemake.sbatch "
      f"{FP_DIR} {FP_OUT}/dia-quant-output/report.pg_matrix.tsv")

In [ ]:
# search database: human proteome + all lincorfz ORFs of this run (one entry per distinct sequence)
def build_search_fasta(ranked_tsv, human_fasta, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    o = pd.read_csv(ranked_tsv, sep="\t", usecols=["orf_label", "orf_id", "locus_name", "ribo_evidence", "aa_seq"],
                    low_memory=False)
    o["aa_seq"] = o["aa_seq"].str.rstrip("*")
    o = o.loc[o["aa_seq"].str.len() >= 7]
    g = o.groupby("aa_seq", sort=False)["orf_label"].apply(list).reset_index()
    g["entry"] = [f"lincORF_{i:05d}" for i in range(1, len(g) + 1)]
    g.explode("orf_label")[["entry", "orf_label"]].merge(o[["orf_label", "orf_id", "locus_name", "ribo_evidence"]]) \
        .to_csv(out_dir / "lincORF_entries.tsv", sep="\t", index=False)
    with open(out_dir / "human_plus_lincORFs.fasta", "w") as fh:
        fh.write(open(human_fasta).read().rstrip("\n") + "\n")
        for r in g.itertuples():
            fh.write(f">{r.entry} {';'.join(r.orf_label[:5])} OS=Homo sapiens GN={r.orf_label[0].split('_')[1]}\n{r.aa_seq}\n")
    return len(o), len(g)

if not (PROT_DIR / "fasta" / "lincORF_entries.tsv").exists():
    # UniProt human proteome (canonical, no decoys/contaminants; the FASTA of the fragpipe-snakemake workflow)
    print(build_search_fasta(RES / "final/lincORFs.ranked.tsv", PROT_DIR / "fasta" / "uniprot_human_2026.fasta",
                             PROT_DIR / "fasta"))

In [ ]:
import bisect, re

def read_fasta(path):
    seqs, name = {}, None
    for line in open(path):
        if line.startswith(">"):
            name = line[1:].split()[0]
            seqs[name] = []
        elif name:
            seqs[name].append(line.strip())
    return {k: "".join(v) for k, v in seqs.items()}

IL = str.maketrans("I", "L")
db = read_fasta(FP_OUT / "fasta" / "database_decoy_contam.fasta")       # as searched (incl. contaminants)
db = {k: v for k, v in db.items() if not k.startswith("rev_")}
linc_ids = [k for k in db if k.startswith("lincORF_")]
canon = "#".join(v.translate(IL) for k, v in db.items() if not k.startswith("lincORF_"))
linc_cat = "#".join(db[k].translate(IL) for k in linc_ids)
linc_starts = np.cumsum([0] + [len(db[k]) + 1 for k in linc_ids[:-1]])

def linc_hits(pep):
    pep, hits, i = pep.translate(IL), set(), 0
    while (i := linc_cat.find(pep, i)) != -1:
        hits.add(linc_ids[bisect.bisect_right(linc_starts, i) - 1])
        i += 1
    return hits

# 1) MSFragger-DIA / Philosopher PSMs (1% FDR), one psm.tsv per experiment
psm_files = [p for p in FP_OUT.rglob("psm.tsv") if "mzMLs" not in p.parts]
psm = pd.concat([pd.read_csv(p, sep="\t", usecols=["Peptide", "Protein", "Mapped Proteins"], low_memory=False)
                 .assign(experiment=p.parent.name) for p in psm_files], ignore_index=True) if psm_files else pd.DataFrame()
# 2) DIA-NN precursor matrix (1% q-value), quantities per run
pr = pd.read_csv(FP_OUT / "dia-quant-output" / "report.pr_matrix.tsv", sep="\t", low_memory=False)
run_cols = [c for c in pr.columns if any(r in c for r in RAW_RUNS)]
print(len(psm_files), "psm.tsv files,", len(psm), "PSMs;", len(pr), "DIA-NN precursors;", len(run_cols), "runs")

prot_cols = [c for c in ["Protein.Ids", "All Mapped Proteins"] if c in pr.columns]
cand = set(pr.loc[pr[prot_cols].astype(str).apply(lambda s: s.str.contains("lincORF_")).any(axis=1), "Stripped.Sequence"])
if len(psm):
    m = psm["Protein"].astype(str).str.contains("lincORF_") | psm["Mapped Proteins"].astype(str).str.contains("lincORF_")
    cand |= set(psm.loc[m, "Peptide"])
rows = []
for pep in sorted(cand):
    sub = pr.loc[pr["Stripped.Sequence"] == pep, run_cols]
    rows.append({"peptide": pep, "lincORF_entries": ",".join(sorted(linc_hits(pep))),
                 "in_canonical_or_contaminant": pep.translate(IL) in canon,
                 "n_psms": int((psm["Peptide"] == pep).sum()) if len(psm) else 0,
                 "n_runs_diann": int(sub.notna().any(axis=0).sum()) if len(sub) else 0})
peps = pd.DataFrame(rows, columns=["peptide", "lincORF_entries", "in_canonical_or_contaminant", "n_psms", "n_runs_diann"])
# stricter: also absent from every GENCODE CDS translation (all isoforms), from the peptide -> RNA step (section 7.2)
src = PROT_DIR / "peptide_genome_mapping" / "reads" / "peptides.tsv"
if src.exists():
    s_ = pd.read_csv(src, sep="\t")
    peps["in_GENCODE_CDS"] = peps["peptide"].isin(set(s_.loc[s_["source_types"].astype(str).str.contains("GENCODE"), "peptide"]))
    peps["in_canonical_or_contaminant"] |= peps["in_GENCODE_CDS"]
peps.to_csv(PROT_ANA / "lincORF_peptides.tsv", sep="\t", index=False)
print(len(peps), "identified peptides map to lincORF entries;",
      int((~peps["in_canonical_or_contaminant"]).sum()), "are absent from canonical/contaminant proteins")
display(peps.loc[~peps["in_canonical_or_contaminant"]].sort_values("n_runs_diann", ascending=False).head(30))

In [ ]:
entries = pd.read_csv(PROT_DIR / "fasta" / "lincORF_entries.tsv", sep="\t")   # linked by orf_id (labels carry the rank)
uniq = peps.loc[~peps["in_canonical_or_contaminant"] & (peps["lincORF_entries"] != "")].copy()
uniq["entry"] = uniq["lincORF_entries"].str.split(",")
uniq = uniq.explode("entry")
per_entry = uniq.groupby("entry").agg(n_unique_peptides=("peptide", "nunique"),
                                      unique_peptides=("peptide", lambda s: ";".join(sorted(set(s)))),
                                      n_psms=("n_psms", "sum"), max_runs_diann=("n_runs_diann", "max")).reset_index()
shared = peps.loc[peps["in_canonical_or_contaminant"] & (peps["lincORF_entries"] != "")].assign(
    entry=lambda d: d["lincORF_entries"].str.split(",")).explode("entry").groupby("entry")["peptide"].nunique() \
    .rename("n_peptides_shared_with_canonical").reset_index()

orfs_p = pd.read_csv(RES / "final/lincORFs.ranked.tsv", sep="\t", low_memory=False)
keep = ["rank", "orf_label", "orf_id", "locus_name", "gene_name_source", "ribo_evidence", "priority_score", "n_aa",
        "resolved_multimapper_dominated", "frac_psites_unique", "MRL", "bayes_log10BF_pooled", "pseudogene_names"]
pep_ev = (orfs_p[[c for c in keep if c in orfs_p]].merge(entries[["entry", "orf_id"]], on="orf_id", how="left")
          .merge(per_entry, on="entry", how="left").merge(shared, on="entry", how="left"))
for c in ["n_unique_peptides", "n_psms", "max_runs_diann", "n_peptides_shared_with_canonical"]:
    pep_ev[c] = pep_ev[c].fillna(0).astype(int)
pep_ev.to_csv(PROT_ANA / "lincORF_peptide_evidence.tsv", sep="\t", index=False)

summ = pep_ev.groupby("ribo_evidence").agg(ORFs=("orf_label", "size"),
                                           with_unique_peptide=("n_unique_peptides", lambda s: int((s > 0).sum())),
                                           only_shared_peptides=("n_peptides_shared_with_canonical",
                                                                 lambda s: int((s > 0).sum())))
summ = summ.reindex([e for e in ["strong", "moderate", "weak", "none"] if e in summ.index])
display(summ)
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
(summ["with_unique_peptide"] / summ["ORFs"] * 100).plot.bar(ax=ax, color="0.3")
ax.set(xlabel="Ribo-seq evidence", ylabel="% ORFs with a unique peptide", title="HepG2 DMEM proteomics (3 runs)")
fig.tight_layout()
fig.savefig(FIG / "proteomics_unique_peptides_by_evidence.pdf")
display(pep_ev.loc[pep_ev["n_unique_peptides"] > 0].sort_values(["max_runs_diann", "n_unique_peptides"],
                                                                 ascending=False).head(40))

## 7.2 All identified peptides on the genome (IGV)

Every peptide identified in the three HepG2 runs (MSFragger PSMs and DIA-NN precursors, 1% FDR; not only the
lincRNA-specific ones) is converted into the RNA sequence(s) that encode it: all GENCODE v42 CDSs and lincorfz
ORFs whose translation contains the peptide (I = L) are collected, and every distinct encoding nucleotide
sequence becomes one read (`peptide_mapping/peptides_to_rna_reads.py`; read name `<peptide>|v<k>|<n sources>`).
A full 3' adapter is appended so that the adapter trimming of the alignment workflow removes exactly the adapter.
One read library is written per proteomics run (the peptides detected in that run).

The reads are aligned with the Ribo-seq alignment workflow of the RNA_oxidation repository
(`NF_pre-rRNA/splicing_trimmed_alignment_all_samples.nf`: cutadapt, STAR Local alignment on the same
annotation-aware index as the Ribo-seq, minimum-edit-distance filtering, probabilistic resolution of multimappers
weighted by unique-read counts per gene). The workflow is used unchanged except for its hard-coded sample names and
sample-sheet checksum, which are patched in a working copy (`PROVENANCE.txt` there). Its fixed 22-nt minimum read
length drops the 7-residue peptides (21 nt).

The resolved BAMs are added to a **new** IGV session (`lincorfz.igv_session.with_peptides.xml`, next to the
pipeline's session, which is left unchanged).

In [ ]:
import sys
PEP_MAP = PROT_DIR / "peptide_genome_mapping"
GENOME_FA, GENCODE_GTF = file_paths["human_genome_file"], file_paths["human_annotation_file"]
MAP_SCRIPTS = PROJECT_DIR / "peptide_mapping"
RUNS_STR = ",".join(RAW_RUNS)
# bash (SLURM): peptides -> RNA reads (one FASTQ per proteomics run) + sample sheet of the alignment workflow
print(f"mkdir -p {PEP_MAP}/reads {PEP_MAP}/slurm_logs && sbatch --job-name=pep2rna --qos=30min --time=00:30:00 "
      f"--cpus-per-task=1 --mem=8G --output={PEP_MAP}/slurm_logs/pep2rna_%j.out --error={PEP_MAP}/slurm_logs/pep2rna_%j.err "
      f"--wrap='PYTHONNOUSERSITE=1 {CONDA_ENVS_DIR}/lincorfz_py/bin/python {MAP_SCRIPTS}/peptides_to_rna_reads.py "
      f"--psm {FP_OUT}/psm.tsv --pr-matrix {FP_OUT}/dia-quant-output/report.pr_matrix.tsv --runs {RUNS_STR} "
      f"--gtf {GENCODE_GTF} --genome {GENOME_FA} --orfs {RES}/final/lincORFs.ranked.tsv --outdir {PEP_MAP}/reads'")
# bash: working copy of NF_pre-rRNA (git clone of zavolanlab/RNA_oxidation, branch juan/nextflow-clean-smrna) in
# {PEP_MAP}/NF_pre-rRNA with patched sample names/checksum and peptides.config (outdir, samplesheet + sha256,
# workDir, conda cacheDir of the original Ribo-seq run, --qos matching the task time); then (SLURM):
print(f"sbatch --output={PEP_MAP}/slurm_logs/driver_%j.out --error={PEP_MAP}/slurm_logs/driver_%j.err "
      f"{MAP_SCRIPTS}/run_peptide_alignment.sbatch {PEP_MAP}/NF_pre-rRNA {PEP_MAP}/NF_pre-rRNA/peptides.config {PEP_MAP}/run")

In [ ]:
pep_tab = pd.read_csv(PEP_MAP / "reads" / "peptides.tsv", sep="\t")
print(len(pep_tab), "peptides;", int((pep_tab["n_sources"] > 0).sum()), "with an encoding sequence;",
      int((pep_tab["peptide"].str.len() < 8).sum()), "of length 7 (below the 22-nt read minimum)")
display(pep_tab["source_types"].fillna("none").value_counts().to_frame("peptides"))
display(pep_tab["n_encodings"].clip(upper=10).value_counts().sort_index().to_frame("peptides by n distinct encodings (10 = >=10)"))
summ = []
for f in sorted((PEP_MAP / "results" / "resolution").glob("*.resolver.summary.tsv")):
    d = pd.read_csv(f, sep="\t", header=None, names=["metric", "value"], comment="#").set_index("metric")["value"]
    summ.append(d.rename(f.name.split(".")[0]))
if summ:
    keep = ["input_read_groups", "single_minimum_NM_passthrough_groups", "probabilistically_resolved_tied_groups",
            "output_alignment_records"]
    display(pd.concat(summ, axis=1).loc[lambda d: d.index.isin(keep)])

In [ ]:
# new IGV session: the pipeline's tracks + the resolved peptide BAMs (the pipeline's session is not modified)
tracks = pd.read_csv(RES / "final" / "igv_tracks.tsv", sep="\t", dtype=str)
bams = sorted((PEP_MAP / "results" / "resolution").glob("*.resolved.bam"))
pep_rows = pd.DataFrame([{"panel": "alignment", "group": "-", "name": f"peptides {b.name.split('.')[0]} (resolved)",
                          "path": str(b), "color": "120,60,160", "range": "-", "height": "60"} for b in bams])
tracks_pep = pd.concat([tracks, pep_rows], ignore_index=True)
tracks_pep.to_csv(PEP_MAP / "igv_tracks.with_peptides.tsv", sep="\t", index=False)
genome_attr = re.search(r'genome="([^"]+)"', open(RES / "final" / "lincorfz.igv_session.xml").read()).group(1)
session_pep = RES / "final" / "lincorfz.igv_session.with_peptides.xml"
subprocess.run([sys.executable, str(LINCORFZ_DIR / "bin" / "make_igv_session.py"), "--genome", genome_attr,
                "--tracks", str(PEP_MAP / "igv_tracks.with_peptides.tsv"), "--out", str(session_pep)], check=True)
print(len(bams), "peptide BAMs added ->", session_pep)

## 7.3 Non-tryptic peptides: HLA class I immunopeptidomes of HepG2 and Huh7

Standard (tryptic) proteomics rarely detects peptides of lincRNA ORFs beyond pseudogenes (section 7.1). No public
small-protein peptidomics data exist for HepG2/Huh7 (PRIDE/MassIVE searched 2026-10-29; the lncRNA-SEP enrichment
dataset PXD019486 has other cell lines), but two HLA class I immunopeptidomes do. HLA-I ligands are non-tryptic
8-12-mers sampled from all translation products, including short-lived micropeptides, and are an established source
of lncRNA-derived peptides:

| dataset | cells | runs |
|:---|:---|:---|
| PXD036232 (= PXD027531; Nat Cell Biol 2023) | HepG2, W6/32 pull-down; control, ArgRS and SRRM2 knock-down | 9 DDA (QE HF-X) |
| PXD053264 (Sci Adv 2024) | Huh7 and Huh7 + HLA-C*w1, W6/32 pull-down | 17 DDA (QE); DIA runs not used |

Search: `fragpipe-snakemake` with the FragPipe template `Nonspecific-HLA-customDB-groupFDR` (non-specific
digestion, 7-25 aa, no carbamidomethylation, cysteinylation variable, MSBooster + Percolator, protein FDR 1% as
set by the template) against UniProt human (`PE=1`) plus all lincorfz ORFs (`PE=2`). **Group FDR**: canonical and
lincRNA-ORF peptides are filtered to 1% FDR separately, so that novel peptides do not ride on the canonical FDR.
A lincRNA-ORF peptide counts as evidence only if it is absent from canonical/contaminant proteins and from all
GENCODE CDS translations (I = L). As in 7.1, this is an annotation; scores and ranking are unchanged.

In [ ]:
HLA_DIR = OUT_BASE / "peptidomics"
HLA_SETS = {"HepG2_MHCI_PXD036232": "HepG2", "Huh7_HLAI_PXD053264": "Huh7"}
HLA_ANA = HLA_DIR / "analysis"
HLA_ANA.mkdir(parents=True, exist_ok=True)
# bash (VS Code/login node, internet): raw files (DDA only) listed in peptidomics/metadata/download_list.tsv
print(f"while IFS=$'\\t' read -r out url size; do curl -s --fail --retry 5 -C - -o $out $url; done "
      f"< {HLA_DIR}/metadata/download_list.tsv")
# search FASTA: UniProt human with PE=1 + lincorfz ORFs with PE=2 (group FDR), built like build_search_fasta()
# bash (SLURM), one fragpipe-snakemake working copy per dataset (config.yaml: Peptidome_DDA,
# Nonspecific-HLA-customDB-groupFDR, input_format raw):
for ds in HLA_SETS:
    print(f"sbatch --output={HLA_DIR}/{ds}/slurm_logs/driver_%j.out --error={HLA_DIR}/{ds}/slurm_logs/driver_%j.err "
          f"{SBATCH_DIR}/run_fragpipe_snakemake.sbatch {HLA_DIR}/{ds}/fragpipe-snakemake {HLA_DIR}/{ds}/fragpipe_out/combined_peptide.tsv")

In [ ]:
hla_psm = []
for ds, cell in HLA_SETS.items():
    for f in sorted((HLA_DIR / ds / "fragpipe_out").glob("*/psm.tsv")):
        if f.parent.name == "mzMLs":
            continue
        hla_psm.append(pd.read_csv(f, sep="\t", usecols=["Peptide", "Protein", "Mapped Proteins", "Spectrum File"],
                                   low_memory=False).assign(dataset=ds, cell=cell, run=f.parent.name))
hla_psm = pd.concat(hla_psm, ignore_index=True)
hla_psm["library"] = np.select([hla_psm["run"].str.contains("PSHC"), hla_psm["run"].str.contains("Huh7_Cw1")],
                               ["HepG2_HLAI", "Huh7Cw1_HLAI"], "Huh7_HLAI")
display(hla_psm.groupby("library").agg(runs=("run", "nunique"), PSMs=("Peptide", "size"), peptides=("Peptide", "nunique")))

# lincRNA-ORF specificity: absent from canonical/contaminant proteins (searched DB) and from all GENCODE CDSs
hla_db = read_fasta(HLA_DIR / "HepG2_MHCI_PXD036232" / "fragpipe_out" / "fasta" / "database_decoy_contam.fasta")
hla_db = {k.split("|")[1] if k.startswith(("sp|", "tr|")) else k: v for k, v in hla_db.items() if not k.startswith("rev_")}
canon_hla = "#".join(v.translate(IL) for k, v in hla_db.items() if not k.startswith("lincORF_"))
hla_linc_ids = [k for k in hla_db if k.startswith("lincORF_")]
hla_linc_cat = "#".join(hla_db[k].translate(IL) for k in hla_linc_ids)
hla_linc_starts = np.cumsum([0] + [len(hla_db[k]) + 1 for k in hla_linc_ids[:-1]])

def hla_linc_hits(pep):
    pep, hits, i = pep.translate(IL), set(), 0
    while (i := hla_linc_cat.find(pep, i)) != -1:
        hits.add(hla_linc_ids[bisect.bisect_right(hla_linc_starts, i) - 1])
        i += 1
    return hits

hla_src = pd.read_csv(HLA_DIR / "peptide_genome_mapping" / "reads" / "peptides.tsv", sep="\t")
in_gencode_hla = set(hla_src.loc[hla_src["source_types"].astype(str).str.contains("GENCODE"), "peptide"])
pep_hla = hla_psm.groupby("Peptide").agg(libraries=("library", lambda s: ",".join(sorted(set(s)))),
                                         n_psms=("run", "size"), n_runs=("run", "nunique")).reset_index()
pep_hla["length"] = pep_hla["Peptide"].str.len()
pep_hla["lincORF_entries"] = [",".join(sorted(hla_linc_hits(p))) for p in pep_hla["Peptide"]]
pep_hla["in_canonical_or_GENCODE"] = [p.translate(IL) in canon_hla or p in in_gencode_hla for p in pep_hla["Peptide"]]
pep_hla["lincORF_specific"] = (pep_hla["lincORF_entries"] != "") & ~pep_hla["in_canonical_or_GENCODE"]
pep_hla.to_csv(HLA_ANA / "immunopeptides.tsv", sep="\t", index=False)
print(len(pep_hla), "peptides;", int((pep_hla["lincORF_entries"] != "").sum()), "contained in lincORF entries;",
      int(pep_hla["lincORF_specific"].sum()), "lincORF-specific")

fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
for lab, sub in [("canonical", pep_hla.loc[~pep_hla["lincORF_specific"]]), ("lincORF-specific", pep_hla.loc[pep_hla["lincORF_specific"]])]:
    ax.hist(sub["length"], bins=np.arange(6.5, 26.5), density=True, histtype="step", lw=2, label=f"{lab} (n={len(sub)})")
ax.set(xlabel="peptide length (aa)", ylabel="density", title="HLA-I ligands: 9-mer peak expected")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "immunopeptide_length.pdf")

In [ ]:
# NOTE: lincORF entries of the peptidomics FASTA are numbered independently of section 7.1 (own entries table)
hla_entries = pd.read_csv(HLA_DIR / "fasta" / "lincORF_entries.tsv", sep="\t")
spec = pep_hla.loc[pep_hla["lincORF_specific"]].assign(entry=lambda d: d["lincORF_entries"].str.split(",")).explode("entry")
per_entry_hla = spec.groupby("entry").agg(n_hla_peptides=("Peptide", "nunique"),
                                          hla_peptides=("Peptide", lambda s: ";".join(sorted(set(s)))),
                                          hla_libraries=("libraries", lambda s: ",".join(sorted(set(",".join(s).split(","))))),
                                          hla_psms=("n_psms", "sum")).reset_index()
orf_cols = ["rank", "orf_label", "orf_id", "locus_name", "gene_name_source", "ribo_evidence", "priority_score",
            "n_aa", "resolved_multimapper_dominated", "frac_psites_unique", "MRL", "bayes_log10BF_pooled", "pseudogene_names"]
hla_ev = (orfs_p[[c for c in orf_cols if c in orfs_p]].merge(hla_entries[["entry", "orf_id"]], on="orf_id", how="left")
          .merge(per_entry_hla, on="entry", how="left"))
hla_ev["n_hla_peptides"] = hla_ev["n_hla_peptides"].fillna(0).astype(int)
hla_ev["pseudogene"] = hla_ev["pseudogene_names"].notna()
hla_ev.to_csv(HLA_ANA / "lincORF_immunopeptide_evidence.tsv", sep="\t", index=False)
sup = hla_ev.loc[hla_ev["n_hla_peptides"] > 0]
display(pd.crosstab(sup["ribo_evidence"], sup["pseudogene"].map({True: "pseudogene", False: "non-pseudogene"}), margins=True))
print(sup["locus_name"].nunique(), "loci with lincORF-specific HLA-I peptides;",
      sup.loc[~sup["pseudogene"], "locus_name"].nunique(), "of them not overlapping pseudogenes")
display(sup.loc[~sup["pseudogene"]].sort_values(["ribo_evidence", "hla_psms"], ascending=[True, False])
        [["rank", "orf_label", "locus_name", "gene_name_source", "ribo_evidence", "resolved_multimapper_dominated",
          "n_aa", "n_hla_peptides", "hla_peptides", "hla_libraries", "hla_psms"]].head(60))

In [ ]:
# all immunopeptides on the genome: same RNA_oxidation alignment workflow as in 7.2 (libraries HepG2_HLAI,
# Huh7_HLAI, Huh7Cw1_HLAI; working copy in peptidomics/peptide_genome_mapping/NF_pre-rRNA)
HLA_MAP = HLA_DIR / "peptide_genome_mapping"
hla_psm_files = " ".join(str(f) for ds in HLA_SETS for f in sorted((HLA_DIR / ds / "fragpipe_out").glob("*/psm.tsv")))
print(f"sbatch --job-name=pep2rna_hla --qos=30min --time=00:30:00 --cpus-per-task=1 --mem=8G "
      f"--output={HLA_MAP}/slurm_logs/pep2rna_%j.out --error={HLA_MAP}/slurm_logs/pep2rna_%j.err "
      f"--wrap='PYTHONNOUSERSITE=1 {CONDA_ENVS_DIR}/lincorfz_py/bin/python {MAP_SCRIPTS}/peptides_to_rna_reads.py "
      f"--psm {hla_psm_files} --runs HepG2_HLAI=PSHC,Huh7_HLAI=Huh7_W632,Huh7Cw1_HLAI=Huh7_Cw1 "
      f"--gtf {GENCODE_GTF} --genome {GENOME_FA} --orfs {RES}/final/lincORFs.ranked.tsv --outdir {HLA_MAP}/reads'")
print(f"sbatch --output={HLA_MAP}/slurm_logs/driver_%j.out --error={HLA_MAP}/slurm_logs/driver_%j.err "
      f"{MAP_SCRIPTS}/run_peptide_alignment.sbatch {HLA_MAP}/NF_pre-rRNA {HLA_MAP}/NF_pre-rRNA/peptides.config {HLA_MAP}/run")
summ = []
for f in sorted((HLA_MAP / "results" / "resolution").glob("*.resolver.summary.tsv")):
    d = pd.read_csv(f, sep="\t", header=None, names=["metric", "value"], comment="#").set_index("metric")["value"]
    summ.append(d.rename(f.name.split(".")[0]))
if summ:
    display(pd.concat(summ, axis=1).loc[lambda d: d.index.isin(["input_read_groups", "single_minimum_NM_passthrough_groups",
                                                                "probabilistically_resolved_tied_groups"])])

# second new IGV session: pipeline tracks + immunopeptide BAMs (+ the tryptic peptide BAMs of 7.2)
hla_bams = sorted((HLA_MAP / "results" / "resolution").glob("*.resolved.bam"))
hla_rows = pd.DataFrame([{"panel": "alignment", "group": "-", "name": f"HLA-I peptides {b.name.split('.')[0]} (resolved)",
                          "path": str(b), "color": "200,90,0", "range": "-", "height": "60"} for b in hla_bams])
tracks_hla = pd.concat([tracks_pep, hla_rows], ignore_index=True)
tracks_hla.to_csv(HLA_MAP / "igv_tracks.with_immunopeptides.tsv", sep="\t", index=False)
session_hla = RES / "final" / "lincorfz.igv_session.with_immunopeptides.xml"
subprocess.run([sys.executable, str(LINCORFZ_DIR / "bin" / "make_igv_session.py"), "--genome", genome_attr,
                "--tracks", str(HLA_MAP / "igv_tracks.with_immunopeptides.tsv"), "--out", str(session_hla)], check=True)
print(len(hla_bams), "immunopeptide BAMs (+", len(bams), "tryptic) ->", session_hla)

## 8. Execute this notebook as a SLURM job

To (re-)generate all outputs and figures of this notebook without computing on a login/VS Code node.

In [ ]:
nb_path = PROJECT_DIR / "Liver_lincRNA_ORFs.ipynb"
logs = OUT_BASE / "slurm_logs"
logs.mkdir(exist_ok=True)
print(f"sbatch --output={logs}/notebook_%j.out --error={logs}/notebook_%j.err "
      f"{SBATCH_DIR}/execute_notebook.sbatch {nb_path} {CONDA_ENVS_DIR / 'lincorfz_py'}")